In [ ]:
# import warnings
# warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.font_manager")

import importlib
import json
import re, os, itertools
from collections import defaultdict
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from fusion_bench.utils.json import load_from_json
import matplotlib.ticker as ticker

# from plot_utils import TASK_TO_LABEL_MAPPING
# from plot_utils import v2_colors as COLORS
# from plot_utils import v2_colors as COLORS_light
# from plot_utils import extra_darker_v2_colors as COLORS_dark

# from matplotlib import colormaps
# COLORS = colormaps['tab10'].colors

plt.rcParams["font.family"] = "Times New Roman"
# plt.rcParams["font.family"] = "DejaVu Serif"
plt.rcParams["mathtext.fontset"] = "cm"

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

PROJECT_ROOT = Path(
    os.path.abspath(
        os.path.join(importlib.import_module("fusion_bench").__path__[0], "..")
    )
)

In [ ]:
import json
from pathlib import Path
import pandas as pd

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def compute_bwt_and_acc(version_dir: str) -> pd.DataFrame:
    """
    For a given version directory (e.g., version_0), compute:
      - average accuracy from `report.json`
      - BWT using all `report_*.json` files (last task accuracy diff)
    Returns:
        pd.DataFrame with columns: ['version', 'acc', 'bwt'] in percentage
    """
    version_dir = Path(version_dir)
    reports = sorted(version_dir.glob("report_*.json"), key=lambda x: int(x.stem.split("_")[1]))

    bwt_sum = 0
    bwt_n = 0

    for i, report_path in enumerate(reports[:-1]):  # exclude final
        data_i = load_json(report_path)
        data_T = load_json(reports[-1])  # final merged model

        # Identify last task in report_i
        task_keys = [k for k in data_i.keys() if k not in {"model_info", "average"}]
        last_task = task_keys[i]

        if not len(task_keys)==i+1: 
            print(f'task num: {len(task_keys)}, report_path: {report_path}')

        acc_i = data_i[last_task]["accuracy"]
        acc_T = data_T[last_task]["accuracy"]

        bwt_sum += acc_T - acc_i
        bwt_n += 1

    bwt = bwt_sum / bwt_n if bwt_n > 0 else 0

    # Load average acc from final report.json
    acc_data = load_json(version_dir / "report.json")
    acc = acc_data["average"]["accuracy"]

    return pd.DataFrame([{
        "version": version_dir.name,
        "acc": acc * 100,
        "bwt": bwt * 100
    }])


def compute_bwt_acc_trf(version_dir: str) -> pd.DataFrame:
    """
    For a given version directory (e.g., version_0), compute:
      - average accuracy (ACC)
      - backward transfer (BWT)
      - forward transfer to unseen tasks (Transfer)
    Returns:
        pd.DataFrame with columns: ['version', 'acc', 'bwt', 'transfer'] in percentage
    """
    version_dir = Path(version_dir)
    reports = sorted(
        version_dir.glob("report_*.json"),
        key=lambda x: int(x.stem.split("_")[1])
    )
    T = len(reports)
    
    # Load final merged model report for ACC and BWT reference
    data_T = load_json(reports[-1])
    
    # Compute ACC from report.json (average accuracy)
    acc_data = load_json(version_dir / "report.json")
    acc = acc_data["average"]["accuracy"]
    
    # Initialize sums for BWT and Transfer
    bwt_sum = 0
    transfer_sum = 0
    
    for i, report_path in enumerate(reports[:-1]):  # Exclude the final report for merging phases
        data_i = load_json(report_path)
        
        # Identify task keys (exclude metadata)
        task_keys = [k for k in data_i.keys() if k not in {"model_info", "average"}]
        if not len(task_keys)==T: 
            print(f'task num: {T}, report_path: {report_path}')
        
        # --- BWT: Last seen task is task i (0-indexed)
        last_task = task_keys[i]
        bwt_sum += (data_T[last_task]["accuracy"] - data_i[last_task]["accuracy"])
        
        # --- Transfer: average performance on unseen tasks
        unseen_keys = task_keys[i+1:]
        if unseen_keys:
            transfer_i = sum(data_i[k]["accuracy"] for k in unseen_keys) / len(unseen_keys)
            transfer_sum += transfer_i
    
    # Finalize metrics
    bwt = bwt_sum / (T - 1) if T > 1 else 0
    transfer = transfer_sum / (T - 1) if T > 1 else 0
    
    # Return a DataFrame with all metrics in percentage
    return pd.DataFrame([{
        "version": version_dir.name,
        "acc": acc * 100,
        "bwt": bwt * 100,
        "trf": transfer * 100
    }])



In [ ]:
# 定义主目录和子实验路径
base_dir = Path("/data1/zihuanqiu/nufilt/outputs")
exp_names = [
    "nufilt",
    # "knots",
]

summary_results = []

for exp in exp_names:
    version_root = base_dir / exp / "vit-b-32-TA8_weight_10_1"
    version_dirs = sorted(version_root.glob("version_*"))
    # version_dirs = [version_root / "version_1"]
    if not version_dirs:
        print(f"⚠️ No version_* found in: {version_root}")
        continue

    # 聚合多个 version
    df = pd.concat([compute_bwt_and_acc(v) for v in version_dirs], ignore_index=True)
    print(df)

    # 提取统计量（均值和标准差）
    acc_mean = df["acc"].mean()
    acc_std = df["acc"].std()
    bwt_mean = df["bwt"].mean()
    bwt_std = df["bwt"].std()

    summary_results.append({
        "exp": exp,
        "acc": acc_mean,
        "acc_std": acc_std,
        "bwt": bwt_mean,
        "bwt_std": bwt_std,
    })

# 最终结果：每个 exp 一行，包含 acc/bwt 的平均与 acc 的标准差
final_df = pd.DataFrame(summary_results)
print(final_df)